# Synthetic Parkfield — end-to-end `codameter` demo

This notebook generates a **10-year synthetic Parkfield-like dv/v dataset** by forward-modelling three environmental contributions:

| Forcing | Model | Reference |
|---|---|---|
| Thermoelastic | 50-day-lagged phase-shift annual cycle | Berger 1975; Okubo et al. 2024 |
| Hydrological | Groundwater-level proxy from precipitation | Roeloffs 1988; Okubo et al. 2024 Eq. 4 |
| Seismic damage | Logarithmic healing transient (M~6 at year 4) | Snieder et al. 2017 |

The synthetic signal is then fed through the full six-phase `codameter` pipeline as if it were real ambient-noise data, and we verify that the truth amplitudes are recovered within ~4σ.

> **No external data required.** Everything is self-contained and runs in ~5 seconds.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (12, 4)})

from codameter import Site, run_workflow
from codameter.config import (
    AnalysisConfig, ForcingSpec, Forcings, Layer, Location,
    MaterialProperties, Measurement, Prior, VelocityModel,
)
from codameter.forward.damage import snieder_healing
from codameter.forward.poroelastic import groundwater_level_okubo
from codameter.forward.thermoelastic import thermoelastic_dvv

YEAR_S = 365.25 * 86400.0
print(f"codameter imported OK — Python {sys.version.split()[0]}")

## 1 · Build the Parkfield site configuration

In [ ]:
site = Site(
    site_id="parkfield_synthetic",
    location=Location(lat=35.97, lon=-120.55, elevation_m=350.0),
    measurement=Measurement(
        type="cross_correlation", frequency_band_hz=(0.9, 1.2),
    ),
    velocity_model=VelocityModel(
        layers=[
            Layer(thickness_km=0.10, vp=1.5, vs=0.6, rho=1.9),
            Layer(thickness_km=0.67, vp=2.5, vs=1.2, rho=2.2),
            Layer(thickness_km=1.00, vp=4.5, vs=2.5, rho=2.5),
            Layer(thickness_km=50.0, vp=5.8, vs=3.4, rho=2.7),
        ],
        source="jeppson_tobin_2015",
    ),
    forcings=Forcings(
        thermoelastic=ForcingSpec(
            enabled=True, model="phase_shift",
            extra={"time_shift_days": 50.0},
        ),
        hydrological=ForcingSpec(enabled=True, model="okubo_gwl"),
        damage=ForcingSpec(enabled=True, model="snieder_healing"),
    ),
    material_properties=MaterialProperties(
        beta_prior=Prior(mean=240.0, std=80.0),
        mu_prime_prior=Prior(mean=250.0, std=90.0),
    ),
    analysis=AnalysisConfig(
        start_date="2010-01-01", end_date="2020-01-01",
        uncertainty_method="wls",
    ),
)
print(site)

## 2 · Generate synthetic dv/v and forcings

Truth amplitudes we want to recover later:

| Parameter | Symbol | Truth value |
|---|---|---|
| GWL sensitivity | `p1_dGWL` | −3.0 × 10⁻³ |
| Thermoelastic sensitivity | `p2_T` | +8.0 × 10⁻⁵ |
| Co-seismic drop | `s_eq` | −2.0 × 10⁻³ |

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
n = 10 * 365
times = pd.date_range("2010-01-01", periods=n, freq="D", tz="UTC")
t_s = (times - times[0]).total_seconds().to_numpy()

# Forcings
T = 15.0 + 8.0 * np.sin(2 * np.pi * t_s / YEAR_S - 0.5) + 0.3 * rng.standard_normal(n)
P = np.zeros(n)
storm = (np.arange(n) % 365) > 300
P[storm] = rng.lognormal(mean=-3.0, sigma=1.5, size=int(storm.sum()))

# Earthquake at year 4
eq_time = times[int(4 * 365)]
eq_t_s = float((eq_time - times[0]).total_seconds())

# Truth amplitudes
truth = {"p1_dGWL": -3.0e-3, "p2_T": 8.0e-5, "s_eq": -2.0e-3}

# Forward-model components
dGWL = groundwater_level_okubo(P, t_s, porosity=0.05, decay_rate_per_s=1.0 / (180 * 86400.0))
dGWL_centred = dGWL - dGWL.mean()
T_pred = thermoelastic_dvv(T, t_s, sensitivity_amplitude=1.0, time_shift_days=50.0)
elapsed = t_s - eq_t_s
healing = snieder_healing(elapsed, tau_min_s=86400.0, tau_max_s=30 * YEAR_S)
L0 = -np.log(30 * YEAR_S / 86400.0)
healing_norm = healing / L0

dvv_clean = (
    truth["p1_dGWL"] * dGWL_centred
    + truth["p2_T"] * T_pred
    + truth["s_eq"] * healing_norm
)
sigma = 1.5e-4
dvv = dvv_clean + sigma * rng.standard_normal(n)

dvv_df = pd.DataFrame({"dvv": dvv, "dvv_err": np.full(n, sigma)}, index=times)
forcings = {
    "precipitation": pd.Series(P, index=times, name="precipitation"),
    "temperature":   pd.Series(T, index=times, name="temperature"),
}
print(f"{len(dvv_df)} daily samples  |  {dvv_df.index[0]:%Y-%m-%d} → {dvv_df.index[-1]:%Y-%m-%d}")
print(f"truth: p1={truth['p1_dGWL']:+.2e}  p2={truth['p2_T']:+.2e}  s_eq={truth['s_eq']:+.2e}")

## 3 · Visualise the synthetic input data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)

axes[0].plot(times, dvv * 1e3, color="steelblue", lw=0.6, alpha=0.8)
axes[0].axvline(eq_time, color="crimson", lw=1.2, ls="--", label=f"M~6 eq ({eq_time:%Y-%m-%d})")
axes[0].set_ylabel("dv/v (\u2030)")
axes[0].set_title("Synthetic dv/v (noise + three forcing contributions)")
axes[0].legend(fontsize=9)

axes[1].plot(times, T, color="orangered", lw=0.8)
axes[1].set_ylabel("Temperature (\u00b0C)")
axes[1].set_title("Daily surface temperature")

axes[2].bar(times, P, color="royalblue", width=1.0, alpha=0.7)
axes[2].set_ylabel("Precipitation (m)")
axes[2].set_title("Daily precipitation")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator())

fig.tight_layout()
plt.show()

## 4 · Run the six-phase `codameter` workflow

In [ ]:
result = run_workflow(dvv_df, forcings, site, earthquake_times=[eq_time])
print(result.summary())

## 5 · Diagnostic six-panel figure

In [ ]:
fig = result.plot_phases()
plt.show()

## 6 · Recovery check — truth vs fitted amplitudes

In [ ]:
fit = result.phase4.fit
for name, true_val in [("p1_dGWL", truth["p1_dGWL"]), ("p2_T", truth["p2_T"])]:
    m, s = fit.posterior.marginal(name)
    z = (m - true_val) / s
    flag = "\u2705 OK" if abs(z) < 4 else "\u274c FAIL"
    print(f"  {name:<10s}  truth={true_val:+.3e}  fit={m:+.3e} \u00b1 {s:.2e}  z={z:+.2f}  {flag}")

print(f"\nchi\u00b2_red = {fit.chi2_reduced:.3f}  (ideal \u2248 1.0; range 0.7\u20131.4 is good)")

In [ ]:
# Bar-chart of recovered parameters with 1-sigma error bars
params = ["p1_dGWL", "p2_T"]
truth_vals = [truth[p] for p in params]
fit_means, fit_sigs = zip(*[fit.posterior.marginal(p) for p in params])

x = np.arange(len(params))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - 0.18, truth_vals, 0.32, label="Truth", color="steelblue", alpha=0.85)
ax.bar(x + 0.18, fit_means, 0.32, label="Fitted", color="tomato", alpha=0.85,
       yerr=fit_sigs, capsize=5, error_kw={"elinewidth": 1.5})
ax.set_xticks(x)
ax.set_xticklabels(params, fontsize=11)
ax.set_ylabel("Amplitude")
ax.set_title("Truth vs. recovered amplitudes (error bars = 1\u03c3)")
ax.legend()
ax.axhline(0, color="k", lw=0.6, ls="--")
fig.tight_layout()
plt.show()

## 7 · Export artefacts to disk

In [ ]:
out_dir = Path("runs/parkfield_synthetic")
out_dir.mkdir(parents=True, exist_ok=True)
result.export(out_dir)
print(f"Artefacts written to {out_dir.resolve()}")
for p in sorted(out_dir.iterdir()):
    print(f"  {p.name}")